## Part1 transformer structure

### word split

In [ ]:
from transformers import AutoTokenizer, AutoModel

# 以 BERT 分词器为例
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "unhappiness is common"
tokens = tokenizer.tokenize(text)
print("Tokens:", tokens)
# Tokens: ['un', '##happy', '##ness', 'is', 'common']

ids = tokenizer.convert_tokens_to_ids(tokens)
print("Token IDs:", ids)

# embedding 过程在模型内部
model = AutoModel.from_pretrained("bert-base-uncased")
inputs = tokenizer(text, return_tensors="pt")
embeddings = model.get_input_embeddings()(inputs["input_ids"])
print("Embedding shape:", embeddings.shape)
# (1, 序列长度, embedding维度768)


/opt/anaconda3/envs/llm/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokens: ['un', '##ha', '##pp', '##iness', 'is', 'common']
Token IDs: [4895, 3270, 9397, 9961, 2003, 2691]
Embedding shape: torch.Size([1, 8, 768])


### self attention

In [2]:
import torch
import torch.nn.functional as F

def self_attention(x):
    d = x.size(-1)
    Q, K, V = x, x, x
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d ** 0.5)
    attn = F.softmax(scores, dim=-1)
    return torch.matmul(attn, V)

x = torch.randn(2, 4, 8)  # batch=2, seq_len=4, dim=8
print(x)

out = self_attention(x)
print(out.shape)  # torch.Size([2, 4, 8])
print(out)


tensor([[[-1.2585,  0.4033, -1.6918, -0.6695, -0.3732, -0.2444,  0.4858,
           0.4681],
         [ 0.9253,  2.0098,  1.0310,  1.4127,  0.0135, -1.2770, -1.2299,
          -0.8179],
         [-1.5491,  0.4202,  0.2986, -0.8086,  0.3032,  1.2772,  0.2199,
          -0.4298],
         [ 0.2701,  0.8226, -1.0345, -0.8355,  0.4250, -0.6738,  0.8663,
          -0.7537]],

        [[ 0.5564, -1.6722,  0.0439, -0.9713, -2.5634, -0.0711,  0.3608,
           0.4875],
         [-0.1524, -1.6829,  0.8755,  0.4404, -1.0984, -1.3759,  0.5338,
          -0.3650],
         [ 0.9991,  0.6982,  0.1470,  1.1956, -0.2482, -0.9407,  0.4335,
          -0.9573],
         [-0.4377,  1.3624,  0.3745, -0.0657, -1.2027, -0.0873, -0.0233,
           0.0354]]])
torch.Size([2, 4, 8])
tensor([[[-0.9552,  0.5237, -1.2041, -0.6756, -0.1077, -0.1233,  0.4812,
           0.0667],
         [ 0.8930,  1.9762,  0.9837,  1.3572,  0.0199, -1.2497, -1.1824,
          -0.8095],
         [-1.2108,  0.5179, -0.1862, -0.7048

### 前馈神经网络；归一化；残差连接

####  前馈神经网络 FFN

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

# 模拟输入 (batch=2, seq_len=4, d_model=512)
x = torch.randn(2, 4, 512)
ffn = FeedForward()
y = ffn(x)
print(y.shape)  # torch.Size([2, 4, 512])


torch.Size([2, 4, 512])


#### BN compare with LN
1. BN 按列
2. LN 按行

In [4]:
import torch
import torch.nn as nn

x = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

bn = nn.BatchNorm1d(num_features=3, affine=False)  
ln = nn.LayerNorm(normalized_shape=3, elementwise_affine=False)

# ❌ bn.eval()  # 先不要 eval，否则用不到 batch 统计量
bn.train()

print("输入:\n", x)
print("BatchNorm:\n", bn(x))
print("LayerNorm:\n", ln(x))



输入:
 tensor([[1., 2., 3.],
        [4., 5., 6.]])
BatchNorm:
 tensor([[-1.0000, -1.0000, -1.0000],
        [ 1.0000,  1.0000,  1.0000]])
LayerNorm:
 tensor([[-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247]])


### 位置编码PE

Transformer 的 Self-Attention 本质是一个 集合运算，对序列中每个 token 处理方式对称。<br>
1. 正余弦位置编码 (Sinusoidal PE)
2. 可学习位置编码 (Learnable PE)
3. RoPE（旋转位置编码）
4. ALiBi（线性偏置）

In [12]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- 正余弦位置编码（不训练）----
def sinusoidal_position_encoding(seq_len, d_model, device=None):
    device = device or "cpu"
    pe = torch.zeros(seq_len, d_model, device=device)
    position = torch.arange(0, seq_len, device=device).unsqueeze(1)            # [L, 1]
    div_term = torch.exp(torch.arange(0, d_model, 2, device=device)
                         * (-math.log(10000.0) / d_model))                     # [d/2]
    pe[:, 0::2] = torch.sin(position * div_term)                               # 偶数维
    pe[:, 1::2] = torch.cos(position * div_term)                               # 奇数维
    return pe                                                                   # [L, d_model]

# ---- 多头自注意力（使用 PyTorch 内置）----
class MHSA(nn.Module):
    def __init__(self, d_model=256, n_heads=4, dropout=0.0):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads,
                                         dropout=dropout, batch_first=True)
    def forward(self, x, attn_mask=None, key_padding_mask=None):
        # x: [B, L, d]
        # key_padding_mask: [B, L], True 表示该位置被 mask（padding）
        y, attn = self.mha(x, x, x, attn_mask=attn_mask, key_padding_mask=key_padding_mask,
                           need_weights=True)
        return y, attn  # y: [B,L,d], attn: [B, heads, L, L]（新版本返回形状可能略不同）

# ---- FFN（升维→激活→降维），可替换为 GELU/SwiGLU ----
class FFN(nn.Module):
    def __init__(self, d_model=256, d_ff=1024, dropout=0.0):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)                 # 可改：F.relu / SwiGLU 实现
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# ---- Transformer Encoder Block（Pre-Norm）----
class TransformerBlock(nn.Module):
    def __init__(self, d_model=256, n_heads=4, d_ff=1024, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MHSA(d_model, n_heads, dropout)
        self.dropout1 = nn.Dropout(dropout)

        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff, dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None, key_padding_mask=None):
        # Pre-Norm + 残差（Attention）
        h = self.ln1(x)
        attn_out, attn_weights = self.attn(h, attn_mask=attn_mask, key_padding_mask=key_padding_mask)
        x = x + self.dropout1(attn_out)

        # Pre-Norm + 残差（FFN）
        h = self.ln2(x)
        ffn_out = self.ffn(h)
        x = x + self.dropout2(ffn_out)

        return x, attn_weights  # 输出与输入同维度 d_model

# ---- 输入层：Embedding + 位置编码（正余弦）----
class TransformerToy(nn.Module):
    def __init__(self, vocab_size=10000, d_model=256, n_heads=4, d_ff=1024,
                 n_layers=2, max_len=512, dropout=0.0, learnable_pe=False):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.learnable_pe = learnable_pe
        self.max_len = max_len
        if learnable_pe:
            self.pos_emb = nn.Embedding(max_len, d_model)   # 可学习PE
        else:
            self.register_buffer("pe_table", sinusoidal_position_encoding(max_len, d_model))

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.ln_out = nn.LayerNorm(d_model)

    def forward(self, input_ids, key_padding_mask=None, debug=False):
        B, L = input_ids.shape
        x = self.token_emb(input_ids)
        print("embed:", x.shape) if debug else None

        if self.learnable_pe:
            pos_ids = torch.arange(0, L, device=input_ids.device)
            x = x + self.pos_emb(pos_ids)[None, :, :]
        else:
            x = x + self.pe_table[:L][None, :, :]
        print("embed+pe:", x.shape) if debug else None

        attn_last = None
        for li, blk in enumerate(self.blocks):
            h = blk.ln1(x)
            print(f"layer{li}.ln1:", h.shape) if debug else None

            attn_out, attn_w = blk.attn(h, key_padding_mask=key_padding_mask)
            print(f"layer{li}.attn_out:", attn_out.shape) if debug else None
            x = x + blk.dropout1(attn_out)

            h = blk.ln2(x)
            print(f"layer{li}.ln2:", h.shape) if debug else None

            ffn_out = blk.ffn(h)
            print(f"layer{li}.ffn_out:", ffn_out.shape) if debug else None
            x = x + blk.dropout2(ffn_out)
            print(f"layer{li}.res:", x.shape) if debug else None

        y = self.ln_out(x)
        print("out:", y.shape) if debug else None
        return y, attn_last


# ------------------ Demo ------------------
if __name__ == "__main__":
    torch.manual_seed(0)
    B, L, V = 2, 6, 1000
    model = TransformerToy(vocab_size=V, d_model=128, n_heads=4, d_ff=512,
                           n_layers=2, max_len=128, dropout=0.1, learnable_pe=False)

    # 构造输入与 padding mask（True=要屏蔽）
    input_ids = torch.randint(0, V, (B, L))
    key_padding_mask = torch.zeros(B, L).bool()   # 这里无 padding，可把后两位置 True 试试

    out, attn = model(input_ids, key_padding_mask=key_padding_mask, debug=True)
    print("输出形状:", out.shape)                 # [B, L, d_model]
    # attn 可能是 [B, heads, L, L] 或 [B, L, L]（取决于 PyTorch 版本）
    
    


embed: torch.Size([2, 6, 128])
embed+pe: torch.Size([2, 6, 128])
layer0.ln1: torch.Size([2, 6, 128])
layer0.attn_out: torch.Size([2, 6, 128])
layer0.ln2: torch.Size([2, 6, 128])
layer0.ffn_out: torch.Size([2, 6, 128])
layer0.res: torch.Size([2, 6, 128])
layer1.ln1: torch.Size([2, 6, 128])
layer1.attn_out: torch.Size([2, 6, 128])
layer1.ln2: torch.Size([2, 6, 128])
layer1.ffn_out: torch.Size([2, 6, 128])
layer1.res: torch.Size([2, 6, 128])
out: torch.Size([2, 6, 128])
输出形状: torch.Size([2, 6, 128])


### 代表模型

encoder-only, decoder-only, encoder-decoder, prefix-decoder等结构及其代表模型，分别适用于哪些任务，为什么现在的大模型都是decoder-only结构

### 解码策略

top-k、top-p、temperature等参数含义，greedy search、beam search等解码策略，投机解码及其优化算法

|策略|主要作用|优点|缺点|
|:----:|:----:|:----:|:----:|
|温度|创造力|灵活，可以动态调整“惊喜度”|温度太高容易产生不连贯的文本|
Top-K	|排除低概率词	|简单有效，避免离谱错误	|K值固定，不够灵活
Top-P	|动态排除低概率词	|智能，能根据概率分布自动调整候选集大小	|比Top-K计算稍复杂
拒绝采样	|强制执行特定规则	|控制力强，可用于避免重复、保证安全等	|规则设定需要经验，可能导致生成变慢

## Part2 主流LLM

### BERT series （Encoder only）理解式模型

以往预训练要么是单向语言模型（如 GPT，左到右），要么是浅层拼接的双向（如 ELMo 的 L2R+R2L），这限制了在句级与词级任务上对双向上下文的充分利用。BERT 用 MLM 让 Transformer 编码器在所有层看到左右文，再加上 NSP 学会句对关系，从而统一地迁移到 NLI、QA、NER 等多类任务（同一架构，仅换输出头即可）。这也是其“统一架构、端到端微调”路线的关键。

In [ ]:
# for bert base

### GPT （Decoder only）

无监督预训练 + 有监督微调（大量训练数据提高模型）

但一般需要高计算性能的设备

### Llama （Decoder only）

https://syhya.github.io/zh/posts/2025-04-06-llama/

- RMS Normalization (RMSNorm)
- FFN_SwiGLU
    - SwiGLU 在引入额外门控（双线性投影）的同时，通过把隐藏层宽度从 4d 调整为 ≈ 2.67d，保证参数量和计算量与传统 FFN 层相当，因此能在不增加成本的前提下提升模型性能。
- Grouped Query Attention (GQA)
- Rotary Positional Embeddings (RoPE)
- Mixture-of-Experts (MoE)

### Qwen （Decoder only）

https://blog.csdn.net/weixin_59191169/article/details/148560050

### GLM

综合学习

### Baichuan （Decoder only）

类似于 Llama 和 Qwen

### Deepseek （Decoder only）

## Part3 Pre-train

大规模训练的高质量数据集/数据清洗与处理方式

## Part4 Post-train

有监督微调 与 对齐
SFT + DPO

### SFT

微调数据构造，数据配比，全参微调，冻结微调，PEFT高效微调 <br>
prompt tuning、p-tuning v2、prefix-tuningadapter-tuning、LoRA及其变体<br>
CoT, Reasoning等o1系列策略 <br>


prefix; adapter; Lora

### PEFT

#### prompt tuning -https://zhuanlan.zhihu.com/p/618871247


Pattern（Template）<br>
即上文提到的Template，其为额外添加的带有[mask]标记的短文本，通常一个样本只有一个Pattern（因为我们希望只有1个让模型预测的[mask]标记）。上文也提到，不同的任务、不同的样本可能会有其更加合适的pattern，因此 如何构建合适的pattern是Prompt-Tuning的研究点之一 ；<br>
Verbalizer <br>
即标签词的映射，对于具体的分类任务，需要选择指定的标签词（label word）。例如情感分析中，我们期望Verbalizer可能是，（positive和negative是类标签）。同样，不同的任务有其相应的label word，但需要注意的是，Verbalizer的构建需要取决于对应的Pattern。因此 如何构建Verbalizer是另一个研究挑战。<br>
上述两个组件被称为Pattern-Verbalizer-Pair（PVP），一般记作 ，在后续的大多数研究中均采用这种PVP组件。

### RLHF & Aligning

为什么有SFT还需要RLHF，两者有何区别，RLAIF，
ReFT, OpenAI做RLHF的过程，里面的几个模型
分别是怎么运作的，DPO的原理和实现 <br>
PPO和DPO对比，DPO有哪些问题以及如何优化，
SimPO, KTO, ORPO, GRPO等

## Part5 模型压缩与量化

- 剪纸
- 蒸馏
- 参数共享
- 架构improve

- 数值表示精度降低
  - Post-Training Quantization (PTQ)
  - Quantization-Aware Training (QAT)
  - 混合精度 (Mixed Precision)
  - QLoRA（量化 + LoRA 微调）


## Part6 专家模型MoE

## Part7 RAG & Agent

## Part8 部署 & 分布式训练 & 推理加速

## Part9 模型评估

## Part10 其他结构